<a href="https://colab.research.google.com/github/rockyanil97-del/aiml/blob/main/Ollama_AI_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Chatbot Using Ollama and Python

**Run this notebook in Google Colab.** It installs Ollama in the Colab runtime, downloads the small `qwen2.5:0.5b` model, and launches a browser-based chatbot using Gradio.

### How to use
1. Upload this `.ipynb` file to [Google Colab](https://colab.research.google.com/).
2. Choose **Runtime → Run all** (or run each cell from top to bottom).
3. Wait for the model download to finish.
4. Open the Gradio link shown in the final cell and chat with your AI.

**Note:** The Colab runtime must remain active while you use the chatbot. The first model download needs an internet connection and may take a few minutes.

## 1. Install the Python packages
This installs Gradio for the chat interface and Requests for calling Ollama's API.

In [1]:
%pip -q install gradio requests

## 2. Install Ollama
This cell downloads and installs Ollama inside the Colab runtime.

## 3. Start the Ollama server
The server runs in the background and provides the local API used by the chatbot.

In [2]:
import time
import requests
import subprocess

OLLAMA_URL = 'http://127.0.0.1:11434'
MODEL_NAME = 'qwen2.5:0.5b'

# Ensure Ollama is installed in the system
try:
    subprocess.run(['ollama', '--version'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print('Ollama is already installed.')
except (subprocess.CalledProcessError, FileNotFoundError):
    print('Installing zstd (dependency)...')
    subprocess.run('sudo apt-get update && sudo apt-get install -y zstd', shell=True, check=True)
    print('Installing Ollama...')
    install_result = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, text=True, capture_output=True)
    if install_result.returncode != 0:
        print(install_result.stderr)
        raise RuntimeError('Ollama installation failed.')
    print('Ollama installed successfully.')

# Start the server only if it is not already responding.
try:
    requests.get(f'{OLLAMA_URL}/api/tags', timeout=2).raise_for_status()
    print('Ollama server is already running.')
except requests.RequestException:
    ollama_log = open('/tmp/ollama_server.log', 'w')
    ollama_process = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=ollama_log,
        stderr=subprocess.STDOUT
    )

    for attempt in range(60):
        try:
            requests.get(f'{OLLAMA_URL}/api/tags', timeout=2).raise_for_status()
            print('Ollama server is ready.')
            break
        except requests.RequestException:
            time.sleep(2)
    else:
        raise RuntimeError('Ollama server did not start. Check /tmp/ollama_server.log for details.')

Ollama is already installed.
Ollama server is ready.


## 4. Download the AI model
This downloads Qwen 2.5 (0.5B parameters), a small model chosen to keep the project lightweight. Run this cell once; subsequent runs may reuse the runtime's downloaded model.

In [3]:
import subprocess

pull_result = subprocess.run(
    ['ollama', 'pull', MODEL_NAME],
    text=True
)
if pull_result.returncode != 0:
    raise RuntimeError('Model download failed. Check your internet connection and run this cell again.')
print(f'{MODEL_NAME} is ready.')

qwen2.5:0.5b is ready.


## 5. Create the chatbot
The chatbot sends your message and the conversation history to Ollama, then displays the model's reply.

In [4]:
import gradio as gr
import requests

# --- CUSTOMIZE YOUR BOT HERE ---
# Change this text to alter the AI's behavior, persona, or guidelines.
SYSTEM_PROMPT = (
    "You are a helpful AI assistant. Answer clearly and use simple English. "
    "When explaining technical topics, give a short example when useful."
)
# Example alternatives:
# SYSTEM_PROMPT = "You are a friendly pirate assistant. Speak like a pirate in all your responses!"
# SYSTEM_PROMPT = "You are an encouraging coding coach. Explain programming topics step-by-step."
# -------------------------------

def chat_with_ollama(message, history):
    # Keep the conversation context so follow-up questions make sense.
    conversation = [{'role': 'system', 'content': SYSTEM_PROMPT}]

    for item in history or []:
        if isinstance(item, dict) and item.get('role') in ('user', 'assistant'):
            content = item.get('content', '')
            if isinstance(content, str) and content.strip():
                conversation.append({'role': item['role'], 'content': content})

    conversation.append({'role': 'user', 'content': message})

    response = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model': MODEL_NAME,
            'messages': conversation,
            'stream': False
        },
        timeout=300
    )
    response.raise_for_status()
    result = response.json()
    return result['message']['content']

print('Chatbot code is ready with customizable System Prompt.')

Chatbot code is ready with customizable System Prompt.


## 6. Launch the chatbot
Run this cell and open the public Gradio URL it prints. Keep this Colab notebook running while chatting.

In [ ]:
demo = gr.ChatInterface(
    fn=chat_with_ollama,
    title='AI Chatbot Using Ollama',
    description='A simple AI chatbot powered by Python, Ollama, and Qwen 2.5.',
    examples=[
        'What is Artificial Intelligence?',
        'Explain Python in simple words.',
        'What is Machine Learning?'
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://46f2b8afe768273301.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Project explanation (for demonstration)

**Objective:** To build a simple AI chatbot that answers questions using a locally running language model.

**Technologies:** Python, Ollama, Qwen 2.5, Gradio, and the Ollama Chat API.

**Working:** The user enters a question in the Gradio chat interface. Python sends the question and chat history to Ollama's API. The Qwen model generates a response, which is displayed in the chat interface.

**Limitation:** The model runs inside the Colab session. The chatbot stops working when that runtime disconnects, and the public Gradio link is temporary.